# Packages and Data Instantiation

In [10]:
import pandas as pd
from pathlib import Path
from typing import Dict

def l_dir(d_p: str = "../data") -> Dict[str, pd.DataFrame]:
    """Loads all CSV files from a directory efficiently into memory.

    Args:
        d_p: Directory path containing the target files.

    Returns:
        Dictionary mapping filename stems to loaded DataFrames.
    """
    return {f.stem: pd.read_csv(f, engine="pyarrow") for f in Path(d_p).glob("*.csv")}

In [11]:
d_m = l_dir()

df_crs = d_m.get("bhp_crs")
df_evt = d_m.get("bhp_evt")
df_ff = d_m.get("bhp_ff")
df_ibs = d_m.get("bhp_ibs")
df_main = d_m.get("bhp_main")

In [12]:
class EDA:
    """Exploratory Data Analysis diagnostics for time-series feature matrices."""

    def __init__(self, m: pd.DataFrame):
        if not isinstance(m, pd.DataFrame):
            raise ValueError("The input 'm' must be a pandas DataFrame. The variable may have been overwritten or failed to load.")
        self.m = m
        self.c = m.columns

    def s_chk(self) -> pd.DataFrame:
        """Calculates column sparsity and missingness."""
        n = self.m.isnull().sum()
        p = (n / len(self.m)) * 100
        t = self.m.dtypes
        return pd.DataFrame({'n_ms': n, 'p_ms': p, 'd_ty': t}).sort_values('p_ms', ascending=False)

    def t_chk(self, d_c: str) -> Dict[str, str]:
        """Validates temporal continuity and identifies boundary limits."""
        dt = pd.to_datetime(self.m[d_c])
        return {
            'strt': str(dt.min().date()),
            'end': str(dt.max().date()),
            'n_dys': str(dt.nunique()),
            'gaps': str(len(dt) - dt.nunique())
        }

    def tgt_sts(self, t_c: str) -> pd.DataFrame:
        """Computes distribution statistics for the target variable."""
        if t_c not in self.c:
            return pd.DataFrame()
        return self.m[[t_c]].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [13]:
eda = EDA(df_main)
eda.s_chk()

,n_ms,p_ms,d_ty
keydeveventtypeid,4750,72.641077,object
headline,4750,72.641077,object
dlycaldt,0,0.000000,object
dlyret,0,0.000000,float64
at,0,0.000000,float64
lt,0,0.000000,float64
dlyvol,0,0.000000,float64
curcd,0,0.000000,object
meanest_fy1,0,0.000000,float64
meanest_fy2,0,0.000000,float64


In [14]:
eda.t_chk('dlycaldt')

{'strt': '2000-01-03', 'end': '2025-12-31', 'n_dys': '6539', 'gaps': '0'}

In [15]:
eda.tgt_sts('dlyret')

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
dlyret,6539.0,0.000714,0.023102,-0.171496,-0.061996,-0.035346,-0.011361,0.000993,0.013135,0.035248,0.05984,0.183184


# Econometrics

In [16]:
# import numpy as np
# import pandas as pd
# from dask import compute, delayed
# import statsmodels.api as sm
# from statsmodels.tsa.stattools import adfuller, kpss, bds
# from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox, het_white, het_breuschpagan
# from statsmodels.tsa.ar_model import ar_select_order, AutoReg
# from statsmodels.stats.stattools import durbin_watson, jarque_bera
# import warnings
# from statsmodels.tools.sm_exceptions import InterpolationWarning

# class TSDiagnostics:
#     def __init__(self, y: np.ndarray):
#         self.y = np.ascontiguousarray(np.asarray(y, dtype=np.float64).ravel())
#         self.y = self.y[np.isfinite(self.y)]
#         self.dy = np.diff(self.y)
#         self.n = self.y.shape[0]
#         self.dy_ok = self.dy.size > 0 and np.ptp(self.dy) > 0.0

#     @staticmethod
#     def _run_test(func, out_dict, key_map, *args, **kwargs):
#         try:
#             res = func(*args, **kwargs)
#             if res is not None:
#                 for key, idx in key_map.items():
#                     out_dict[key] = res[idx] if isinstance(res, tuple) else res
#         except Exception:
#             pass

#     @delayed
#     def _test_stationarity(self, max_lag: int) -> dict:
#         out = {}
#         self._run_test(adfuller, out, {'adf_stat': 0, 'adf_pval': 1}, self.y, maxlag=max_lag, autolag='AIC')
#         with warnings.catch_warnings():
#             warnings.simplefilter("ignore", InterpolationWarning)
#             self._run_test(kpss, out, {'kpss_stat': 0, 'kpss_pval': 1}, self.y, regression='c', nlags='auto')

#         if self.dy_ok:
#             self._run_test(adfuller, out, {'adf_d_stat': 0, 'adf_d_pval': 1}, self.dy, maxlag=max_lag, autolag='AIC')
#             with warnings.catch_warnings():
#                 warnings.simplefilter("ignore", InterpolationWarning)
#                 self._run_test(kpss, out, {'kpss_d_stat': 0, 'kpss_d_pval': 1}, self.dy, regression='c', nlags='auto')
#         return out

#     @delayed
#     def _test_serial_corr(self, lags: int) -> dict:
#         out = {}
#         try:
#             out['lb_pval'] = acorr_ljungbox(self.y, lags=[lags], return_df=False).iloc[0, 1]
#         except Exception:
#             pass

#         if self.dy_ok:
#             try:
#                 out['lb_d_pval'] = acorr_ljungbox(self.dy, lags=[lags], return_df=False).iloc[0, 1]
#                 out['lb_abs_d_pval'] = acorr_ljungbox(np.abs(self.dy), lags=[lags], return_df=False).iloc[0, 1]
#                 out['lb_d2_pval'] = acorr_ljungbox(self.dy**2, lags=[lags], return_df=False).iloc[0, 1]
#             except Exception:
#                 pass
#             self._run_test(het_arch, out, {'arch_lm_pval': 1}, self.dy, nlags=lags)
#         return out

#     @delayed
#     def _test_residuals(self) -> dict:
#         out = {}
#         xc = np.ones((self.n, 1), dtype=np.float64)
#         xh = np.column_stack((np.ones(self.n, dtype=np.float64), np.arange(self.n, dtype=np.float64)))

#         try:
#             ols = sm.OLS(self.y, xc).fit()
#             r = np.asarray(ols.resid, dtype=np.float64)

#             try:
#                 out["dw_stat"] = float(durbin_watson(r))
#             except Exception:
#                 pass

#             self._run_test(het_white, out, {"hetw_lm_p": 1}, r, xh)
#             self._run_test(het_breuschpagan, out, {"hetbp_lm_p": 1}, r, xh, robust=True)
#             self._run_test(jarque_bera, out, {"jb_p": 1, "jb_skew": 2, "jb_kurt": 3}, r)

#         except Exception:
#             pass

#         if self.dy_ok:
#             try:
#                 s, p = bds(self.dy, max_dim=2)
#                 out.update({"bds_stat": s[0] if isinstance(s, np.ndarray) else s,
#                             "bds_p": p[0] if isinstance(p, np.ndarray) else p})
#             except Exception:
#                 pass
#         return out

#     @delayed
#     def _fit_ar(self, max_lag: int) -> dict:
#         out = {}
#         try:
#             sel = ar_select_order(self.y, maxlag=max_lag, ic='bic', trend='c')
#             lags = sel.ar_lags if (sel.ar_lags is not None and len(sel.ar_lags) > 0) else [1]
#             mod = AutoReg(self.y, lags=lags, trend='c').fit()
#             out.update({'ar_lags': len(lags), 'ar_bic': mod.bic})
#         except Exception:
#             pass

#         if self.dy_ok:
#              try:
#                 sel_d = ar_select_order(self.dy, maxlag=max_lag, ic='bic', trend='c')
#                 lags_d = sel_d.ar_lags if (sel_d.ar_lags is not None and len(sel_d.ar_lags) > 0) else [1]
#                 mod_d = AutoReg(self.dy, lags=lags_d, trend='c').fit()
#                 out.update({'ar_d_lags': len(lags_d), 'ar_d_bic': mod_d.bic})
#              except Exception:
#                 pass
#         return out

#     @classmethod
#     def profile_assets(cls, df: pd.DataFrame, target_col: str, asset_col: str, min_obs: int = 50) -> pd.DataFrame:
#         if asset_col not in df.columns:
#             df = df.copy()
#             df[asset_col] = 'TARGET'

#         tasks = {}
#         for asset, group in df.groupby(asset_col, sort=False):
#             series = group[target_col].to_numpy()
#             if np.isfinite(series).sum() >= min_obs:
#                 engine = cls(series)
#                 dyn_lag = max(1, min(12, engine.n // 10))

#                 t1 = engine._test_stationarity(dyn_lag)
#                 t2 = engine._test_serial_corr(dyn_lag)
#                 t3 = engine._test_residuals()
#                 t4 = engine._fit_ar(dyn_lag)

#                 tasks[asset] = delayed(lambda *dicts: {k: v for d in dicts for k, v in d.items()})(t1, t2, t3, t4)

#         results = compute(tasks)[0]
#         return pd.DataFrame.from_dict(results, orient='index')

In [17]:
# df_econometrics = TSDiagnostics.profile_assets(df_main, target_col='dlyret', asset_col='ticker')
# print(df_econometrics)

# fPCA

In [18]:
import re
import numpy as np
import pandas as pd
from typing import List
from econml.dml import LinearDML
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MultiLabelBinarizer

class CausalEventIsolator:
    """Double Machine Learning pipeline for causal catalyst impact extraction.

    Implements Orthogonalized Machine Learning using the Frisch-Waugh-Lovell
    theorem. Transforms discrete multi-label event occurrences into a
    high-dimensional treatment matrix, and isolates the Average Treatment
    Effect (ATE) vector by partialling out continuous confounding matrices.
    Utilizes multi-output histogram-based gradient boosting for O(1)
    nuisance parameter estimation across all event types simultaneously.

    Time Complexity: O(N * F * T + T * N * log(N))
    Space Complexity: O(N * F + N * T)

    Attributes:
        target_col (str): Dependent variable representing post-event return.
        event_col (str): Column containing string-formatted array of event IDs.
        confounders (List[str]): Exogenous variables to residualize against.
        min_obs (int): Minimum positive treatment instances for matrix invertibility.
        cv_folds (int): Cross-fitting partitions for out-of-sample nuisance estimation.
        random_state (int): Seed for deterministic cross-fitting splits.
        results (pd.DataFrame): The extracted treatment effects and test statistics.
    """

    def __init__(
        self,
        target_col: str,
        event_col: str,
        confounders: List[str],
        min_obs: int = 15,
        cv_folds: int = 5,
        random_state: int = 42
    ):
        """Initializes the DML pipeline configuration.

        Args:
            target_col (str): The column name for the target outcome.
            event_col (str): The column name for the event strings.
            confounders (List[str]): List of continuous covariates.
            min_obs (int): Minimum valid observations per event type.
            cv_folds (int): Number of splits for cross-fitting.
            random_state (int): State seed for reproducible cross-fitting.

        Raises:
            ValueError: If cv_folds < 2 or min_obs < cv_folds.
        """
        if cv_folds < 2:
            raise ValueError("Cross-fitting requires at least 2 folds.")
        if min_obs < cv_folds:
            raise ValueError("Minimum observations must strictly exceed cv_folds.")

        self.target_col = target_col
        self.event_col = event_col
        self.confounders = confounders
        self.min_obs = min_obs
        self.cv_folds = cv_folds
        self.random_state = random_state
        self._mlb = MultiLabelBinarizer()
        self.results = None

    def _encode_treatments(self, df: pd.DataFrame) -> pd.DataFrame:
        """Parses and binarizes string-array event representations.

        Args:
            df (pd.DataFrame): The filtered dataset containing event strings.

        Returns:
            pd.DataFrame: Validated treatment matrix T of shape (N, Valid_Events).
        """
        parsed_events = df[self.event_col].astype(str).str.findall(r'\d+').apply(lambda x: [int(i) for i in x])

        t_matrix = pd.DataFrame(
            self._mlb.fit_transform(parsed_events),
            columns=self._mlb.classes_,
            index=df.index
        )

        counts = t_matrix.sum(axis=0)
        valid_columns = counts[counts >= self.min_obs].index
        return t_matrix[valid_columns]

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Estimates the structural causal impact of all valid events.

        Args:
            df (pd.DataFrame): The raw integrated feature matrix.

        Returns:
            pd.DataFrame: Ranked empirical results containing ATE and T-Statistics.

        Raises:
            KeyError: If mandatory columns are missing from the input matrix.
            RuntimeError: If zero valid event types meet the observation threshold.
        """
        required_cols = [self.target_col, self.event_col] + self.confounders
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Input matrix is missing required columns: {missing}")

        df_clean = df.dropna(subset=required_cols).copy()

        Y = df_clean[self.target_col].to_numpy()
        X = df_clean[self.confounders].to_numpy()

        df_T = self._encode_treatments(df_clean)
        if df_T.empty:
            raise RuntimeError("No event types meet the minimum observation threshold.")

        T = df_T.to_numpy()

        dml = LinearDML(
            model_y=HistGradientBoostingRegressor(random_state=self.random_state),
            model_t=MultiOutputRegressor(HistGradientBoostingRegressor(random_state=self.random_state)),
            discrete_treatment=False,
            cv=self.cv_folds,
            random_state=self.random_state
        )

        dml.fit(Y, T, X=X)

        ate_matrix = dml.ate(X)
        stderr_matrix = dml.ate_stderr(X)

        mean_ate = np.mean(ate_matrix, axis=0) if ate_matrix.ndim > 1 else np.array([np.mean(ate_matrix)])
        mean_stderr = np.mean(stderr_matrix, axis=0) if stderr_matrix.ndim > 1 else np.array([np.mean(stderr_matrix)])

        t_stats = np.divide(
            mean_ate,
            mean_stderr,
            out=np.zeros_like(mean_ate),
            where=(mean_stderr != 0)
        )

        results_df = pd.DataFrame({
            'event_id': df_T.columns,
            'n_occurrences': df_T.sum(axis=0).to_numpy(),
            'ate_mean': mean_ate,
            'ate_stderr': mean_stderr,
            't_stat': t_stats
        })

        self.results = results_df.iloc[results_df['t_stat'].abs().argsort()[::-1]].reset_index(drop=True)
        return self.results

c:\Users\Zac Kienzle\Miniconda3\envs\SIGStockPitch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Estimation

In [19]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import MultiLabelBinarizer

class FeatureMatrix:
    """Constructs exponentially weighted features and forward targets.

    Attributes:
        target_horizon (int): Trading days for forward cumulative abnormal return.
        ewm_span (int): Span for exponential smoothing decay factor.
    """

    def __init__(self, target_horizon: int = 5, ewm_span: int = 20):
        self.target_horizon = target_horizon
        self.ewm_span = ewm_span

    def transform(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, str, List[str]]:
        """Engineers EWMA confounders and discrete forward targets.

        Args:
            df (pd.DataFrame): Base dataset containing returns and market factors.

        Returns:
            Tuple[pd.DataFrame, str, List[str]]: Processed dataframe, target column name,
            and the list of confounder column names.

        Raises:
            KeyError: If required baseline columns are missing.
            ValueError: If the input dataframe is empty.
        """
        if df.empty:
            raise ValueError("Input DataFrame is empty.")

        req_cols = ['dlyret', 'mktrf', 'smb', 'hml']
        missing = [c for c in req_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Missing required columns: {missing}")

        df_out = df.copy()

        df_out['ar'] = df_out['dlyret'] - df_out['mktrf']

        target_col = f'car_fwd_{self.target_horizon}d'
        df_out[target_col] = sum(df_out['ar'].shift(-i) for i in range(1, self.target_horizon + 1))

        df_out['vol_ewm'] = (
            df_out['dlyret']
            .ewm(span=self.ewm_span, adjust=False)
            .std() * np.sqrt(252)
        )

        df_out['mom_ewm'] = (
            df_out['dlyret']
            .ewm(span=self.ewm_span, adjust=False)
            .mean()
        )

        base_confounders = ['mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'vol_ewm', 'mom_ewm']

        opt_cols = [
            'meanest_fy1', 'meanest_fy2', 'highest_fy1', 'highest_fy2',
            'lowest_fy1', 'lowest_fy2', 'at', 'lt', 'dlyvol'
        ]
        confounders = base_confounders + [c for c in opt_cols if c in df_out.columns]

        df_out.dropna(subset=[target_col] + confounders, inplace=True)

        return df_out.reset_index(drop=True), target_col, confounders

In [20]:
class CausalEstimator:
    """Frisch-Waugh-Lovell Double Machine Learning estimator.

    Attributes:
        target_col (str): Outcome variable column name.
        event_col (str): Event array column name.
        confounders (List[str]): Covariates for orthogonalization.
        cv_folds (int): Cross-fitting partitions.
        min_obs (int): Minimum treatments for asymptotic validity.
        random_state (int): Seed for deterministic splits.
        results (pd.DataFrame): Output statistics.
    """

    def __init__(
        self,
        target_col: str,
        event_col: str,
        confounders: List[str],
        cv_folds: int = 5,
        min_obs: int = 15,
        random_state: int = 42
    ):
        if cv_folds < 2:
            raise ValueError("cv_folds must be >= 2.")
        if min_obs < cv_folds:
            raise ValueError("min_obs must exceed cv_folds.")

        self.target_col = target_col
        self.event_col = event_col
        self.confounders = confounders
        self.cv_folds = cv_folds
        self.min_obs = min_obs
        self.random_state = random_state
        self.results: Optional[pd.DataFrame] = None
        self._mlb = MultiLabelBinarizer()

    def _encode_treatments(self, df: pd.DataFrame) -> pd.DataFrame:
        """Binarizes string-array events into a valid treatment matrix.

        Args:
            df (pd.DataFrame): Dataset containing event strings.

        Returns:
            pd.DataFrame: Treatment matrix of shape (N, Valid_Events).
        """
        parsed_events = (
            df[self.event_col]
            .astype(str)
            .str.findall(r'\d+')
            .apply(lambda x: [int(i) for i in x])
        )

        t_matrix = pd.DataFrame(
            self._mlb.fit_transform(parsed_events),
            columns=self._mlb.classes_,
            index=df.index
        )

        valid_columns = t_matrix.columns[t_matrix.sum(axis=0) >= self.min_obs]
        return t_matrix[valid_columns]

    def _orthogonalize(self, X: np.ndarray, y: np.ndarray, t: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Cross-fits nuisance models to extract structural residuals.

        Args:
            X (np.ndarray): Confounder matrix.
            y (np.ndarray): Outcome vector.
            t (np.ndarray): Treatment vector.

        Returns:
            Tuple[np.ndarray, np.ndarray]: Residuals (y_tilde, t_tilde).
        """
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)

        y_resid = np.empty_like(y, dtype=np.float64)
        t_resid = np.empty_like(t, dtype=np.float64)

        for train_idx, test_idx in kf.split(X):
            X_train, X_test = X[train_idx], X[test_idx]

            model_y = HistGradientBoostingRegressor(random_state=self.random_state)
            model_y.fit(X_train, y[train_idx])
            y_resid[test_idx] = y[test_idx] - model_y.predict(X_test)

            model_t = HistGradientBoostingClassifier(random_state=self.random_state)
            if len(np.unique(t[train_idx])) > 1:
                model_t.fit(X_train, t[train_idx])
                t_resid[test_idx] = t[test_idx] - model_t.predict_proba(X_test)[:, 1]
            else:
                t_resid[test_idx] = t[test_idx]

        return y_resid, t_resid

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Executes FWL causal estimation across all valid events.

        Args:
            df (pd.DataFrame): Integrated feature and event matrix.

        Returns:
            pd.DataFrame: ATE, standard errors, and test statistics per event.

        Raises:
            KeyError: If required columns are missing.
            RuntimeError: If no events meet the min_obs threshold.
        """
        req_cols = [self.target_col, self.event_col] + self.confounders
        missing = [c for c in req_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Input missing columns: {missing}")

        df_clean = df.dropna(subset=[self.target_col, self.event_col])

        y = df_clean[self.target_col].to_numpy(dtype=np.float64)
        X = df_clean[self.confounders].to_numpy(dtype=np.float64)

        df_T = self._encode_treatments(df_clean)
        if df_T.empty:
            raise RuntimeError("No event types meet the min_obs threshold.")

        out = []

        for event_id in df_T.columns:
            t = df_T[event_id].to_numpy(dtype=np.float64)
            y_tilde, t_tilde = self._orthogonalize(X, y, t)

            t_tilde_ss = np.dot(t_tilde, t_tilde)
            if t_tilde_ss == 0:
                continue

            ate = np.dot(t_tilde, y_tilde) / t_tilde_ss
            epsilon_hat = y_tilde - (ate * t_tilde)

            variance = np.sum((t_tilde * epsilon_hat) ** 2) / (t_tilde_ss ** 2)
            stderr = np.sqrt(variance)

            t_stat = ate / stderr if stderr > 0 else np.nan
            p_val = 2 * (1 - norm.cdf(np.abs(t_stat))) if not np.isnan(t_stat) else np.nan

            out.append({
                'event_id': event_id,
                'n_occurrences': int(np.sum(t)),
                'ate': ate,
                'hc0_stderr': stderr,
                't_stat': t_stat,
                'p_val': p_val
            })

        df_res = pd.DataFrame(out)
        if not df_res.empty:
            df_res = df_res.iloc[df_res['t_stat'].abs().argsort()[::-1]].reset_index(drop=True)

        self.results = df_res
        return self.results

In [21]:
matrix_builder = FeatureMatrix(target_horizon=5, ewm_span=20)
df_features, target, confounders = matrix_builder.transform(df_main)

estimator = CausalEstimator(
    target_col=target,
    event_col='keydeveventtypeid',
    confounders=confounders,
    cv_folds=5,
    min_obs=15
)

causal_results = estimator.fit_transform(df_features)
print(causal_results.head())

   event_id  n_occurrences       ate  hc0_stderr    t_stat     p_val
0         1             61 -0.012254    0.004562 -2.685837  0.007235
1        42             22  0.019283    0.009616  2.005304  0.044931
2       149            192  0.004859    0.002774  1.751787  0.079810
3        16            202 -0.004621    0.002721 -1.697951  0.089517
4       219             29  0.012839    0.007610  1.687197  0.091566
